In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data,load_test_data

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

# 1. Define the RNN Model
class RNN(nn.Module):
    def __init__(self, input_size, output_size, rnn_units, fc_units, dropout=[0.0, 0.0, 0.0]):
        super(RNN, self).__init__()
        
        self.input_dropout = nn.Dropout(dropout[0])
        self.rnn = nn.GRU(input_size, rnn_units, batch_first=True)
        self.rnn_dropout = nn.Dropout(dropout[1])
        self.fc = nn.Linear(rnn_units, fc_units)
        self.fc_dropout = nn.Dropout(dropout[2])
        self.output = nn.Linear(fc_units, output_size)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.input_dropout(x)
        rnn_out, _ = self.rnn(x)
        rnn_out = self.rnn_dropout(rnn_out[:, -1, :])  # Take the last time step
        fc_out = self.relu(self.fc(rnn_out))
        fc_out = self.fc_dropout(fc_out)
        out = self.output(fc_out)
        return out

# 2. Weighted Huber Loss Function
def weighted_huber_loss(predictions, targets, delta=1.0):
    # Standard Huber Loss
    loss = torch.nn.functional.huber_loss(predictions.squeeze(), targets, delta=delta, reduction='none')
    
    # Apply weights
    weights = torch.ones_like(targets)
    weights[targets.abs() > 2.3] = 4.0   # Medium weight (reduced)
    weights[targets.abs() > 5.0] = 10.0  # Few weight (reduced)
    
    return (weights * loss).mean()

# 3. Training Function
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        preds = model(x)
        loss = nn.L1Loss() (preds.squeeze(), y)  # Using L1 Loss for training
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(dataloader)

# 4. Evaluation Function with Per-Band Analysis
def evaluate_model(model, dataloader, device):
    model.eval()
    results = []
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            preds = model(x).squeeze()
            
            # Store predictions and targets for analysis
            res = pd.DataFrame({
                'target': y.cpu().numpy(),
                'pred': preds.cpu().numpy()
            })
            results.append(res)
            
    eval_df = pd.concat(results)
    eval_df['error_sq'] = (eval_df['target'] - eval_df['pred'])**2
    
    # Categorize into bands for reporting
    def get_band(val):
        if abs(val) > 5.0: return 'Few'
        if abs(val) > 2.3: return 'Medium'
        return 'Many'
    
    eval_df['band'] = eval_df['target'].apply(get_band)
    
    # Calculate Mean Squared Error per band
    band_metrics = eval_df.groupby('band')['error_sq'].mean().to_dict()
    total_mse = eval_df['error_sq'].mean()
    
    return total_mse, band_metrics

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 5. Load Data using our updated strategy
    # Note: Ensure load_data is imported from your dataloading.py
    train_loader, val_loader, pipeline = load_data(
        train_path='../../datasets_challenge/train',
        dist_csv_path='../../datasets_challenge/sideSlip_distribution.csv',
        val_file_name='train_part_017.csv', # Our "Drift Stress Test" file
        columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],
        sequence_length = 20,
        threshold = True,  # Enable velocity thresholding
        threshold_value = 3.0,  # Only keep samples where vehicle speed > 3 m/s
        threshold_column = "vxCG",  # Column to apply threshold on (longitudinal velocity)
        target='sideSlip',
        batchsize=64
    )

    # 6. Initialize Model
    # Set device for computation (CPU or GPU)
    device = "cpu"  # Change to "cuda" if you have a GPU available

    # Model hyperparameters
    input_size = 22  # Number of features (22 in this case)
    rnn_units = 48  # Number of units in RNN hidden state
    fc_units = 48  # Number of units in fully connected layer
    output_size = 1  # Single output (sideslip angle)
    lr = 0.0003  # Learning rate for optimizer

    # Initialize the model with specified architecture
    model = RNN(
        input_size=input_size, 
        output_size=output_size,
        rnn_units=rnn_units,
        fc_units=fc_units,
        dropout=[0.0, 0.0, 0.0]  # Dropout rates for [input, after_rnn, after_fc]
    ).to(device)

    # Define optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)  # Adam optimizer for weight updates

    # 7. Run Training Loop
    epochs = 50
    print(f"{'Epoch':<8} | {'Train Loss':<12} | {'Val MSE':<10} | {'Few-Shot MSE':<12}")
    print("-" * 60)

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_mse, band_mse = evaluate_model(model, val_loader, device)
        
        few_mse = band_mse.get('Few', 0.0)
        
        print(f"{epoch+1:<8} | {train_loss:<12.5f} | {val_mse:<10.5f} | {few_mse:<12.5f}")

    print("\nFinal Performance on Validation Set (train_part_017.csv):")
    for band, mse in band_mse.items():
        print(f"Band: {band:<7} | MSE: {mse:.5f}")

PRUNING: Skipping redundant file train_part_002.csv
PRUNING: Skipping redundant file train_part_014.csv
VALIDATION: Loading train_part_017.csv
Calculating weights for WeightedRandomSampler...
Epoch    | Train Loss   | Val MSE    | Few-Shot MSE
------------------------------------------------------------
1        | 0.54001      | 0.20557    | 0.41464     
2        | 0.27065      | 0.16319    | 0.39190     
3        | 0.23491      | 0.18391    | 0.46917     
4        | 0.21809      | 0.15856    | 0.40419     
5        | 0.20568      | 0.18340    | 0.49064     
6        | 0.19684      | 0.21151    | 0.58395     
7        | 0.18780      | 0.16147    | 0.45190     
8        | 0.18244      | 0.21296    | 0.52850     
9        | 0.17639      | 0.18874    | 0.55165     
10       | 0.16130      | 0.21449    | 0.62995     
11       | 0.15082      | 0.17514    | 0.56403     
12       | 0.14714      | 0.21136    | 0.65318     
13       | 0.14141      | 0.16099    | 0.54183     
14       | 0.13872 

In [2]:
print("\nLoading test data...")
test_dataloader = load_test_data(test_path = "../../datasets_challenge/test",
                                 batchsize = 64,
                                 pipeline = pipeline, 
                                 load_anonymized_data=True)


Loading test data...
../../datasets_challenge/test\test_part_001.csv
../../datasets_challenge/test\test_part_002.csv
../../datasets_challenge/test\test_part_003.csv
../../datasets_challenge/test\test_part_004.csv
../../datasets_challenge/test\test_part_005.csv
../../datasets_challenge/test\test_part_006.csv
../../datasets_challenge/test\test_part_007.csv
../../datasets_challenge/test\test_part_008.csv
../../datasets_challenge/test\test_part_009.csv
../../datasets_challenge/test\test_part_010.csv
../../datasets_challenge/test\test_part_011.csv


In [3]:
# Generate predictions on test set
model.eval()
total_loss = 0  
test_preds = []
test_ground_truths = []  # Will contain placeholder values (all zeros)

with torch.no_grad():
    for step, (x, y) in enumerate(test_dataloader):
        X, y = x.to(device), y.to(device).reshape([len(y), 1])

        pred = model(X)
        loss = nn.L1Loss()(pred, y)
        total_loss += loss.item()
        # Note: loss is meaningless here since y contains placeholder values
        
        # Store predictions
        test_preds.append(pred.reshape(-1).cpu().numpy())
        test_ground_truths.append(y.reshape(-1).cpu().numpy())

    # Concatenate all batches
    test_ground_truths = np.concatenate(test_ground_truths)
    test_preds = np.concatenate(test_preds)
    
print("Shape of Predictions", test_preds.shape)

Shape of Predictions (49328,)


In [4]:
# Save predictions in the required format for submission
y_pred = pd.DataFrame({"predicted_sideslip": test_preds})
y_pred["id"] = range(len(test_preds))  # Add sample IDs (0 to 49327)

# Save to CSV file
y_pred.to_csv("predictionsvalidaation3.csv", index=False)

print(f"✓ Predictions saved to 'predictionsvalidaation3.csv'")
print(f"✓ Total samples: {len(test_preds)}")
print(f"✓ Ready for submission to the competition!")

✓ Predictions saved to 'predictionsvalidaation3.csv'
✓ Total samples: 49328
✓ Ready for submission to the competition!


In [1]:
# 3. Training Function
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        preds = model(x)
        loss = nn.L1Loss() (preds.squeeze(), y)  # Using L1 Loss for training
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    return running_loss / len(dataloader)

# 4. Evaluation Function with Per-Band Analysis
def evaluate_model(model, dataloader, device):
    model.eval()
    results = []
    
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            preds = model(x).squeeze()
            
            # Store predictions and targets for analysis
            res = pd.DataFrame({
                'target': y.cpu().numpy(),
                'pred': preds.cpu().numpy()
            })
            results.append(res)
            
    eval_df = pd.concat(results)
    eval_df['error_sq'] = (eval_df['target'] - eval_df['pred'])**2
    
    # Categorize into bands for reporting
    def get_band(val):
        if abs(val) > 5.0: return 'Few'
        if abs(val) > 2.3: return 'Medium'
        return 'Many'
    
    eval_df['band'] = eval_df['target'].apply(get_band)
    
    # Calculate Mean Squared Error per band
    band_metrics = eval_df.groupby('band')['error_sq'].mean().to_dict()
    total_mse = eval_df['error_sq'].mean()
    
    return total_mse, band_metrics


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import glob
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loadingbins import load_data,load_test_data

# 1. Get all available training files
train_path = '../../datasets_challenge/train'
all_files = sorted([os.path.basename(f) for f in glob.glob(os.path.join(train_path, '*.csv'))])

# 2. Storage for final results
loso_results = []

print(f"Starting LOSO Cross-Validation on {len(all_files)} sessions...")

for val_file in all_files:
    print(f"\n" + "="*50)
    print(f"FOLD: Validating on {val_file}")
    print("="*50)
    
    # 3. Load Data (val_file_name handles the 'Leave-One-Out' logic automatically)
    train_loader, val_loader, _ = load_data(
        train_path=train_path,
        dist_csv_path='../../datasets_challenge/sideSlip_distribution.csv',
        columns_to_drop = ["sideSlip", "vyCG", "time", "longitude","latitude"],
        val_file_name=val_file, 
        sequence_length = 20,
        threshold = True,  # Enable velocity thresholding
        threshold_value = 3.0,  # Only keep samples where vehicle speed > 3 m/s
        threshold_column = "vxCG",  # Column to apply threshold on (longitudinal velocity)
        target='sideSlip',
        batchsize=64
    )
    
    # 1. Define the RNN Model
    class RNN(nn.Module):
        def __init__(self, input_size, output_size, rnn_units, fc_units, dropout=[0.0, 0.0, 0.0]):
            super(RNN, self).__init__()
            
            self.input_dropout = nn.Dropout(dropout[0])
            self.rnn = nn.GRU(input_size, rnn_units, batch_first=True)
            self.rnn_dropout = nn.Dropout(dropout[1])
            self.fc = nn.Linear(rnn_units, fc_units)
            self.fc_dropout = nn.Dropout(dropout[2])
            self.output = nn.Linear(fc_units, output_size)
            self.relu = nn.ReLU()
            
        def forward(self, x):
            x = self.input_dropout(x)
            rnn_out, _ = self.rnn(x)
            rnn_out = self.rnn_dropout(rnn_out[:, -1, :])  # Take the last time step
            fc_out = self.relu(self.fc(rnn_out))
            fc_out = self.fc_dropout(fc_out)
            out = self.output(fc_out)
            return out
    
    # Set device for computation (CPU or GPU)
    device = "cpu"  # Change to "cuda" if you have a GPU available
    
    # 4. Initialize a fresh model for every fold
    model = RNN(input_size=22, output_size=1, rnn_units=64, fc_units=64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    
    # 5. Short Training (e.g., 10-15 epochs per fold to save time)
    best_val_mse = float('inf')
    fold_metrics = {}

    for epoch in range(15):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_mse, band_mse = evaluate_model(model, val_loader, device)
        
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            fold_metrics = band_mse # Store the best band metrics for this file
            
    # 6. Store results for this specific file
    fold_metrics['file'] = val_file
    fold_metrics['total_mse'] = best_val_mse
    loso_results.append(fold_metrics)
    
    print(f"Finished {val_file}. Few-Shot MSE: {fold_metrics.get('Few', 0):.5f}")

# 7. Analyze Final Generalization
loso_df = pd.DataFrame(loso_results)
print("\n" + "!"*50)
print("FINAL LOSO REPORT")
print("!"*50)
print(loso_df[['file', 'Many', 'Medium', 'Few', 'total_mse']])

# Identify the "Hardest" files for the model
hardest_files = loso_df.sort_values(by='Few', ascending=False).head(3)
print("\nHardest files to predict (Highest Few-Shot Error):")
print(hardest_files[['file', 'Few']])

Starting LOSO Cross-Validation on 22 sessions...

FOLD: Validating on train_part_001.csv
VALIDATION: Loading train_part_001.csv
PRUNING: Skipping redundant file train_part_002.csv
PRUNING: Skipping redundant file train_part_014.csv
Calculating weights for WeightedRandomSampler...


RuntimeError: input.size(-1) must be equal to input_size. Expected 22, got 27